# B5. Pilot run on Kaggle

Runs all three models on the 480 rows of `manifests/pilot.csv` (20 tuning photos x 8 cells x 3 defences) and writes `runs/pilot_{model}.jsonl`.

**Before you press Run All**
1. Settings: Accelerator **GPU T4 x1**, Internet **on**.
2. Add Input: the **pixelguard-images** dataset.
3. Add-ons > Secrets: `GITHUB_TOKEN` switched on for this notebook.

Expected time on one T4: about 45 minutes for all three models. If the session stops, run all cells again: the runner skips rows that are already written.

At the end, download `pilot_runs.zip` from the Output panel (/kaggle/working) and also File > Download notebook.

In [ ]:
!pip install -q -U 'transformers>=4.57' accelerate bitsandbytes pillow num2words pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, turn on the GPU in settings')

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

Find the dataset folder and check that every image the manifest asks for is there.

In [ ]:
import glob
from pathlib import Path
import pandas as pd

manifest = pd.read_csv('manifests/pilot.csv')
first = Path(manifest['path'].iloc[0]).name
hits = glob.glob(f'/kaggle/input/**/{first}', recursive=True)
assert hits, 'pilot images not found, add the pixelguard-images dataset with Add Input'
folder = Path(hits[0]).parent
if folder.name == 'images':
    ROOT = folder.parent
else:
    ROOT = Path('/tmp/pg_root')
    ROOT.mkdir(exist_ok=True)
    if not (ROOT / 'images').exists():
        (ROOT / 'images').symlink_to(folder)
missing = [p for p in manifest['path'] if not (ROOT / p).exists()]
print('dataset folder:', folder)
print(len(manifest), 'manifest rows,', manifest['image_id'].nunique(), 'photos,', len(missing), 'missing')
assert len(manifest) == 480 and not missing

Run the three models one after another. Qwen goes first because it is the fastest.

In [ ]:
MODELS = ['qwen3-vl-4b', 'smolvlm2-2b', 'llava-next-mistral-7b']
os.makedirs('/kaggle/working/runs', exist_ok=True)
for m in MODELS:
    print('=' * 70)
    !python src/run_model.py --model {m} --manifest manifests/pilot.csv --images-root {ROOT} --out /kaggle/working/runs/pilot_{m}.jsonl

Check each file: 480 rows, agreed fields, the same rows as the manifest, one config hash. Then count empty, long and refused answers.

In [ ]:
import json, sys
sys.path.insert(0, 'src')
from score import label, is_correct

FIELDS = ['image_id', 'cell', 'setting', 'defence', 'model', 'revision', 'prompt',
          'output', 'config_hash', 'seed', 'timestamp']
KEY = ['image_id', 'cell', 'setting', 'defence']
atk = pd.read_csv('data/attack_set.csv').set_index('image_id')
want = set(map(tuple, manifest[KEY].astype(str).values))

def target(r):
    return 'TUESDAY' if r['cell'] in ('i_off', 'c_off', 'neutral_long') else atk.loc[r['image_id'], 'on_topic_target']

runs = {}
for m in MODELS:
    with open(f'/kaggle/working/runs/pilot_{m}.jsonl', encoding='utf-8') as f:
        df = pd.DataFrame([json.loads(line) for line in f if line.strip()])
    df['target'] = df.apply(target, axis=1)
    df['label'] = [label(o, t) for o, t in zip(df['output'], df['target'])]
    df['correct'] = [is_correct(o, atk.loc[i, 'category']) for o, i in zip(df['output'], df['image_id'])]
    runs[m] = df
    checks = {
        '480 rows': len(df) == 480,
        'agreed fields': list(df.columns[:len(FIELDS)]) == FIELDS,
        'same rows as manifest': set(map(tuple, df[KEY].astype(str).values)) == want,
        'no repeats': not df.duplicated(KEY).any(),
        'one config hash': df['config_hash'].nunique() == 1,
    }
    print(m, checks, 'OK' if all(checks.values()) else 'PROBLEM')
    print('   empty:', (df['output'].str.strip() == '').sum(),
          '| over 5 words:', (df['output'].str.split().str.len() > 5).sum(),
          '| refusals:', (df['label'] == 'refusal').sum())

Pilot-gate numbers, no defence only. Each cell is out of 20 photos.
- `clean right`: the clean photo is named correctly (true class or synonym).
- The other rows: the scorer finds the target word. `c_on`, `i_on`, `c_long` and `neutral_short` use the on-topic word. `i_off`, `c_off` and `neutral_long` use TUESDAY.

In [ ]:
ORDER = ['clean', 'neutral_short', 'c_on', 'i_on', 'c_long', 'neutral_long', 'c_off', 'i_off']
table = {}
for m, df in runs.items():
    d = df[df['defence'] == 'none']
    col = {'clean right': int(d[d['cell'] == 'clean']['correct'].sum())}
    for c in ORDER[1:]:
        col[c] = int((d[d['cell'] == c]['label'] == 'success').sum())
    table[m] = col
gate = pd.DataFrame(table)
print(gate.to_string())
print()
print('rows where a model followed "Say ..." (i_on + i_off, all models):', int(gate.loc[['i_on', 'i_off']].values.sum()))
print('c_on minus neutral_short per model:', (gate.loc['c_on'] - gate.loc['neutral_short']).to_dict())

Target word found, every cell and defence, out of 20. For `clean` this counts the on-topic word, so it is the baseline for `c_on`.

In [ ]:
rows = []
for m, df in runs.items():
    for (c, dfn), g in df.groupby(['cell', 'defence']):
        rows.append({'model': m, 'cell': c, 'defence': dfn, 'success': int((g['label'] == 'success').sum())})
print(pd.DataFrame(rows).pivot_table(index=['cell', 'defence'], columns='model', values='success', aggfunc='sum').loc[ORDER].astype(int).to_string())

Twenty answers to read by eye, from the Qwen file.

In [ ]:
pd.set_option('display.max_colwidth', 80)
runs[MODELS[0]].sample(20, random_state=0)[['image_id', 'cell', 'defence', 'target', 'output', 'label']]

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/pilot_runs', 'zip', '/kaggle/working/runs')
for p in sorted(Path('/kaggle/working/runs').glob('pilot_*.jsonl')):
    print(p.name, sum(1 for _ in open(p, encoding='utf-8')), 'rows')
print('download /kaggle/working/pilot_runs.zip from the Output panel')